# Linear solvers with SIP Poisson

This tutorial solves a small manufactured Poisson problem with the existing `SipPoisson` application. It is intended as a compact place to try the currently configured BoSSS direct solvers and the PETSc Krylov/direct adapters.

The exact solution is quadratic, so the example uses DG degree 2 and checks its L2 error after every successful solve.

## Prerequisites

Build BoSSSpad in Release mode before opening this worksheet:

```bash
dotnet build public/src/L4-application/BoSSSpad/BoSSSpad.csproj -c Release
```

The first code cell uses the matching `BoSSSpad` build output. For a flat local native-library build, set `BOSSS_NATIVE_OVERRIDE` to the directory containing the `libBoSSSnative_*.so` files before starting Jupyter. For example, in WSL:

```bash
export BOSSS_NATIVE_OVERRIDE=$HOME/src/BoSSS-native/lib
```

Use the `.NET (C#)` kernel. Run only one selected solver per kernel session when experimenting with PETSc; restart the kernel before changing a PETSc native variant or MPI/thread launch configuration.

In [ ]:
#r "BoSSSpad.dll"
#r "ipPoisson.dll"

using System;
using System.Collections.Generic;
using System.Data;
using System.Linq;
using System.Text.RegularExpressions;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.SipPoisson;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Control;
using ilPSP.LinSolvers.PETSc;
using SolverCodes = BoSSS.Solution.Control.LinearSolverCode;
using static BoSSS.Application.BoSSSpad.BoSSSshell;

Init();

## Solver catalogue

The selector below exposes these configurations:

- **BoSSS direct solvers:** PARDISO, MUMPS, and UMFPACK.
- **PETSc Krylov solvers:** CG with Jacobi, CG with GAMG, and GMRES with ASM.
- **PETSc direct solvers:** default LU, Cholesky, MUMPS, UMFPACK, and SuperLU_DIST.

Availability depends on the native PETSc variant. `SuperLU_DIST` requires an MPI-capable PETSc variant; the default and UMFPACK PETSc packages are sequential-only. The worksheet reports unavailable or inapplicable optional packages as a skip instead of treating them as a failed Poisson solve.

In [ ]:
var solverCases = new Dictionary<string, ISolverFactory> {
    ["BoSSS PARDISO"] = SolverCodes.direct_pardiso.GetConfig(),
    ["BoSSS MUMPS"] = SolverCodes.direct_mumps.GetConfig(),
    ["BoSSS UMFPACK"] = SolverCodes.direct_umfpack.GetConfig(),

    ["PETSc GMRES + ILU(2)"] = new PETScKrylovSolverConfig {
        NativeVariant = PETScNativeVariant.Auto,
        PrintConvergenceMonitor = true,
        PETScOptions = "-ksp_type gmres -ksp_gmres_restart 100 -pc_type ilu -pc_factor_levels 2 -pc_factor_mat_ordering_type rcm -ksp_rtol 1e-10",
        ConvergenceCriterion = 1.0e-10,
        MaxSolverIterations = 2000
    },
    ["PETSc GMRES + ASM"] = new PETScKrylovSolverConfig {
        NativeVariant = PETScNativeVariant.Auto,
        PrintConvergenceMonitor = true,
        PETScOptions = @"-ksp_type gmres -ksp_gmres_restart 80 -ksp_max_it 200 -ksp_rtol 1e-10 
                     -pc_type asm -pc_asm_overlap 1 -pc_asm_type restrict -sub_ksp_type preonly 
                     -sub_pc_type lu -sub_pc_factor_mat_solver_type umfpack -sub_pc_factor_mat_ordering_type rcm 
                     -ksp_converged_reason -ksp_monitor",
        ConvergenceCriterion = 1.0e-10,
        MaxSolverIterations = 2000
    },
    ["BoSSS multigrid (K-cycle Schwarz)"] = SolverCodes.exp_Kcycle_schwarz.GetConfig(),

    ["PETSc direct LU"] = SolverCodes.direct_petsc.GetConfig(),
    ["PETSc direct Cholesky"] = SolverCodes.direct_petsc_cholesky.GetConfig(),
    ["PETSc direct MUMPS"] = SolverCodes.direct_petsc_mumps.GetConfig(),
    ["PETSc direct UMFPACK"] = SolverCodes.direct_petsc_umfpack.GetConfig(),
    ["PETSc direct SuperLU_DIST"] = SolverCodes.direct_petsc_superlu_dist.GetConfig(),
};

foreach (var entry in solverCases)
    Console.WriteLine(entry.Key);

## Submit a solver comparison as batch jobs

The cells below submit every compatible configuration as an independent BoSSSpad `Job`. This gives each PETSc solver a fresh process and preserves the normal runtime selection. They use the queue configured as the BoSSSpad default. The three degree-2 grids are small (42 by 21, 5,292 scalar unknowns), medium (168 by 84, 84,672 unknowns), and large (336 by 168, 338,688 unknowns).

Each run uses exactly four cores, either one MPI rank with four OpenMP threads (`omp4`) or four MPI ranks with one thread each (`mpi4`). No mixed MPI-plus-OpenMP configuration is submitted because no mixed PETSc binary is installed. Serial-only ILU cases are therefore submitted only as `omp4`.

The jobs save their sessions in a batch-compatible database named `SipPoissonLinearSolverTutorial`. Re-running the submission cell creates new jobs, so change `jobPrefix` when retaining an earlier comparison.

In [ ]:
const string workflowName = "SipPoissonLinearSolverTutorial";
//WorkflowMgmTestUtils.DeleteDatabase(workflowName);
//WorkflowMgmTestUtils.DeleteDeployments(workflowName + "*");
//BoSSSshell.Reset();
BoSSSshell.WorkflowMgm.Init(workflowName);
// The benchmark submits otherwise-equal controls with different MPI/OpenMP modes.
// Match results by the unique job/session name, not by control-object equality.
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();
var batch = GetDefaultQueue();
if (BoSSSshell.WorkflowMgm.DefaultDatabase == null)
    throw new InvalidOperationException("The selected queue could not create a compatible database.");

Console.WriteLine($"Using default queue: {batch}");

In [ ]:
var problemSizes = new[] {
    (Label: "small", XRes: 42, YRes: 21),
    (Label: "medium", XRes: 168, YRes: 84),
    (Label: "large", XRes: 336, YRes: 168),
};

// Every parallel case is MPI-only or OpenMP-only; no mixed MPI-plus-OpenMP job is submitted.
var solverModes = new[] {
    (Solver: "PETSc GMRES + ILU(2)", MpiProcs: 1, Threads: 4, Label: "omp4"),
    (Solver: "PETSc GMRES + ASM", MpiProcs: 4, Threads: 1, Label: "mpi4"),

    // BoSSS direct solvers; PARDISO is included twice to measure OpenMP scaling.
    (Solver: "BoSSS PARDISO", MpiProcs: 1, Threads: 1, Label: "serial"),
    (Solver: "BoSSS PARDISO", MpiProcs: 1, Threads: 4, Label: "omp4"),
    (Solver: "BoSSS MUMPS", MpiProcs: 4, Threads: 1, Label: "mpi4"),
    (Solver: "BoSSS UMFPACK", MpiProcs: 1, Threads: 1, Label: "serial"),
    (Solver: "BoSSS multigrid (K-cycle Schwarz)", MpiProcs: 1, Threads: 4, Label: "omp4"),

    // PETSc direct solvers: sequential packages use the OpenMP PETSc variant, distributed packages use MPI.
    (Solver: "PETSc direct LU", MpiProcs: 1, Threads: 4, Label: "omp4"),
    (Solver: "PETSc direct Cholesky", MpiProcs: 1, Threads: 4, Label: "omp4"),
    (Solver: "PETSc direct UMFPACK", MpiProcs: 1, Threads: 4, Label: "omp4"),
    (Solver: "PETSc direct MUMPS", MpiProcs: 4, Threads: 1, Label: "mpi4"),
    (Solver: "PETSc direct SuperLU_DIST", MpiProcs: 4, Threads: 1, Label: "mpi4"),
};

var batchCases = problemSizes.SelectMany(size => solverModes.Select(mode =>
    (Solver: mode.Solver, MpiProcs: mode.MpiProcs, Threads: mode.Threads, ModeLabel: mode.Label,
     SizeLabel: size.Label, XRes: size.XRes, YRes: size.YRes))).ToArray();

string jobPrefix = "SipPoissonLinearSolverTutorial_small_medium_large_4cores";
var submittedJobs = new List<(string Solver, int MpiProcs, int Threads, Job Job)>();

foreach (var testCase in batchCases) {
    string safeName = string.Concat(testCase.Solver.Select(character =>
        char.IsLetterOrDigit(character) ? character : '_'));
    string jobName = $"{jobPrefix}_{testCase.SizeLabel}_{safeName}_{testCase.ModeLabel}_{testCase.MpiProcs}x{testCase.Threads}";

    var control = SipHardcodedControl.TestCartesian1(xRes: testCase.XRes, yRes: testCase.YRes, pDG: 2);
    control.savetodb = true;
    control.TracingNamespaces = "*";
    control.SetDatabase(BoSSSshell.WorkflowMgm.DefaultDatabase);
    control.LinearSolver = solverCases[testCase.Solver];
    control.ProjectName = $"SipPoissonLinearSolverTutorial/{jobName}";
    control.SessionName = jobName;

    var job = new Job(jobName, typeof(SipPoissonMain)) {
        NumberOfMPIProcs = testCase.MpiProcs,
        NumberOfThreads = testCase.Threads,
        ExecutionTime = "00:20:00"
    };
    job.SetControlObject(control);
    job.Activate(batch);
    submittedJobs.Add(($"{testCase.Solver} ({testCase.SizeLabel}, {testCase.ModeLabel})", testCase.MpiProcs, testCase.Threads, job));
    Console.WriteLine($"Submitted {jobName}.");
}

## Wait and compare

Run this cell after the submission cell. It waits for the short jobs to finish and then displays a small comparison table. A failed or unavailable solver remains visible with its job status; successful jobs include their stored `SolL2err` query.

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(TimeOutSeconds: 1800, PollingIntervallSeconds: 5);

var comparison = new DataTable("SIP Poisson linear solver comparison");
comparison.Columns.Add("Solver");
comparison.Columns.Add("MPI ranks", typeof(int));
comparison.Columns.Add("Threads/rank", typeof(int));
comparison.Columns.Add("Status");
comparison.Columns.Add("Job runtime", typeof(TimeSpan));
comparison.Columns.Add("Linear solve", typeof(TimeSpan));
comparison.Columns.Add("L2 error");
comparison.Columns.Add("Accuracy");
comparison.Columns.Add("Session folder");

foreach (var item in submittedJobs) {
    var session = item.Job.LatestSession;
    string sessionFolder = session == null ? null : System.IO.Path.Combine(session.Database.Path, "sessions", session.ID.ToString());
    int? actualMpiRanks = session?.ComputeNodeNames?.Count;
    int? actualThreads = session?.ThreadPerMPIRank;
    object l2Error = null;
    if (session != null && session.KeysAndQueries.ContainsKey("SolL2err"))
        l2Error = session.KeysAndQueries["SolL2err"];

    var match = Regex.Match(item.Job.Stdout ?? string.Empty, @"SIP_LinearSolver:\s*([0-9.Ee+-]+) sec");
    TimeSpan? linearSolve = match.Success
        ? TimeSpan.FromSeconds(double.Parse(match.Groups[1].Value, System.Globalization.CultureInfo.InvariantCulture))
        : null;

    bool accurate = l2Error is double error && error < 1.0e-6;
    string status = item.Job.Status.ToString();
    if (item.Job.Status == JobStatus.FinishedSuccessful && !accurate)
        status = "INVALID accuracy";

    comparison.Rows.Add(
        item.Solver,
        actualMpiRanks ?? (object)DBNull.Value,
        actualThreads ?? (object)DBNull.Value,
        status,
        item.Job.LatestDeployment?.RunTime,
        linearSolve.HasValue ? linearSolve.Value : DBNull.Value,
        l2Error ?? DBNull.Value,
        accurate ? "OK" : "FAIL",
        sessionFolder ?? (object)DBNull.Value);
}

Console.WriteLine("Solver                                             MPI  Threads/rank  Status                 Job runtime  Linear solve  L2 error          Accuracy  Session folder");
Console.WriteLine(new string('-', 190));
foreach (DataRow row in comparison.Rows) {
    string jobRuntime = row["Job runtime"] == DBNull.Value ? "-" : $"{((TimeSpan)row["Job runtime"]).TotalSeconds:F2} s";
    string linearSolve = row["Linear solve"] == DBNull.Value ? "-" : $"{((TimeSpan)row["Linear solve"]).TotalSeconds:F2} s";
    string l2Error = row["L2 error"] == DBNull.Value ? "-" : row["L2 error"].ToString();
    string sessionFolder = row["Session folder"] == DBNull.Value ? "-" : row["Session folder"].ToString();
    string mpiRanks = row["MPI ranks"] == DBNull.Value ? "-" : row["MPI ranks"].ToString();
    string threadsPerRank = row["Threads/rank"] == DBNull.Value ? "-" : row["Threads/rank"].ToString();
    Console.WriteLine($"{row["Solver"],-50} {mpiRanks,3} {threadsPerRank,13}  {row["Status"],-20} {jobRuntime,-12} {linearSolve,-13} {l2Error,-17} {row["Accuracy"],-9} {sessionFolder}");
}

## Choosing a PETSc runtime

`PETScNativeVariant.Auto` selects `petsc_omp` for one MPI rank and `petsc_mpi` or `petsc_mpi_omp` for multiple ranks, depending on the thread count. The PETSc configurations in this tutorial intentionally use `Auto` so that the same worksheet follows the normal BoSSS selection policy.

For a distributed direct solve, select `PETSc direct MUMPS` or `PETSc direct SuperLU_DIST` and start the notebook kernel in an MPI-capable execution environment. Do not use PETSc default LU or PETSc UMFPACK with multiple MPI ranks.